# PRP plotting and analysis notebook

This notebook loads the saved PRP datasets, subsets the data to selected phonemes/emotions, visualizes PRP waveforms and F-statistics, and runs exploratory Eelbrain statistical tests.

**Main analysis choices in this notebook**
- Sampling rate: 128 Hz
- PRP time window: -0.1 to 1.0 s
- Conditions retained: `hap` and `sad`
- Primary ROI: fronto-central sensors (`AF3`, `AF4`, `Fz`, `F3`, `F4`, `FC1`, `FC2`)

Run the notebook from top to bottom so that shared variables such as `prps`, `fronto_central_chs`, `prp_data`, and `fstats` are defined before later plotting/statistics cells.


## 1. Imports

Load the packages used for PRP handling, Eelbrain plotting/statistics, and custom Matplotlib figures.


In [ ]:
import neuraspeech as ns
import eelbrain
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 2. Load PRP data

Create the `PRPData` object from the saved MATLAB-derived PRP files and attach the BioSemi montage. The summary cell is a quick check that subjects, conditions, manners, and phonemes loaded correctly.


In [ ]:
path_to_data = 'C:/projects/emo_EEG/data_pipeline/visualization_analysis/prp/NH'
path_to_montage_file = 'C:/projects/emo_EEG/data/biosemi_32ch_2mastoid_locs.csv'
fig_out_dir = 'C:/projects/emo_EEG/data_pipeline/visualization_analysis/figures'
stat_out_dir = 'C:/projects/emo_EEG/data_pipeline/visualization_analysis/statistics'

In [ ]:
prps = ns.PRPData(
    data_path = path_to_data,
    sr = 128,
    time_win = (-0.1, 1),
    eeg_fieldname = 'eegs',
    phoneme_fieldname = 'phonemes',
    montage_path = path_to_montage_file
)
prps.data_summary()

## 3. Subset and save the working dataset

This section restricts the dataset to the selected phonemes and emotion conditions. The saved pickle can be reused later without rerunning the full loading/subsetting step.

The optional `manner_filter` is left commented so the notebook keeps all manners after the phoneme/emotion filter unless you explicitly turn it on.


In [ ]:
filter = {'phoneme': ['DH', 'T', 'IH', 'N', 'R', 'OW', 'Z', 'D', 'AW',
                 'L', 'ER', 'K', 'M', 'S', 'HH', 'AA', 'SH', 'EY',
                 'G', 'W', 'P', 'AY', 'AO', 'F', 'UW', 'AE'],
          'cond':['hap', 'sad']
         }
manner_filter = {'manner': ['stop', 'vowel']}
prps.subset_dataset(filter, exclude = False)
# prps.subset_dataset(manner_filter, exclude = False)

In [ ]:
prps.save('C:/projects/emo_EEG/data_pipeline/visualization_analysis/prp/CI/subset/CI_prp.pkl')

## 4. Define the fronto-central ROI

These sensors are used repeatedly for waveform plots and ROI-level summaries.


## 5. PRP waveform plots

These plots show PRP time courses averaged over the fronto-central ROI. Use them first to inspect the overall waveform shape before running statistics.


### 5.1 Grand-average PRP by manner

A high-level view of manner-specific PRP waveforms across the retained data.


In [ ]:
# Fronto-central channels
fronto_central_chs = ['AF3', 'AF4', 'Fz', 'F3', 'F4', 'FC1', 'FC2']

prps.plot_manner_prps(
    split_by= ['task'],
    line_by = 'manner',
    title_prefix = '(FC)',
    subset_sensors = fronto_central_chs
)

In [ ]:
from neuraspeech.config import MANNER_COLORS
ds = prps.get_data()

# IMPORTANT: only subset sensors (NO averaging yet)
ds['PRP'] = ds['PRP'].sub(sensor=fronto_central_chs)

# -------------------------
# SAME aggregation as plot_manner_prps
# -------------------------
group_factors = ['task', 'manner']   # split_by + line_by
agg_by = ' % '.join(group_factors)

drop_vars = [x for x in ds.keys() if x not in group_factors + ['PRP']]
ds = ds.aggregate(agg_by, drop=drop_vars)

# =========================
# Plot (match _plot_time_course)
# =========================
fig, ax = plt.subplots(figsize=(8, 5))

for manner in ds['manner'].cells:

    tmp = ds.sub(f"manner == '{manner}'")

    # there is exactly 1 row per (task, manner)
    nd = tmp['PRP'][0]

    time = nd.time
    time_arr = np.linspace(time.tmin, time.tmax, time.nsamples)

    y = nd.get_data()

    # CRITICAL: average sensors HERE (same as source code)
    if y.shape[-1] > 1:
        y = np.mean(y, axis=-1)

    y = np.squeeze(y)

    ax.plot(
        time_arr, 
        y, 
        label=manner, 
        linewidth=3,
        color=MANNER_COLORS[manner])

# =========================
# Styling (match function)
# =========================
ax.axvline(0, color='k', linestyle='--', linewidth=1)
#ax.axhline(0, color='k', linewidth=1)

ax.set_ylabel('PRP amplitude (µV)', fontsize=14)
ax.set_xlabel('Time (s)', fontsize=14)
ax.set_title('(FC) task=emo', fontsize=14)

ax.legend(
    loc='lower center',
    bbox_to_anchor=(0.5, -0.25),
    ncol=len(ds['manner'].cells),
    frameon=False,
    fontsize=14
)

plt.tight_layout()
fig.savefig(
    f"{fig_out_dir}/NH_prp_manner_plot.png",
    dpi=900,
    bbox_inches="tight"
)
plt.show()

### 5.2 PRP by subject and condition

This plot checks whether the condition-level patterns are consistent across individual subjects.


In [ ]:
# Plot the averaged PRP for each manner, divided by the factors in split_by
prps.plot_manner_prps(
    split_by = ['sub', 'cond'],                       # Factor(s) used to divide the figure. Separate subplot for level of the 'task' factor
    line_by = 'manner',                        # Within each suplot, do separate PRP lines for each manner category (averaged across phonemes of the same manner)
    subset_sensors = ['AF3', 'F3', 'FC1'],       # Plot the average PRPs in this subset of electrodes
    equal_y_scale = True,                       # Ensure that the y-scale is the same across subplots
    show_topo = False         
    )

### 5.3 PRP by condition

This collapses across subjects and separates the curves by condition, giving a cleaner condition-level comparison.


In [ ]:
prps.plot_manner_prps(
    split_by = ['cond'],                       # Factor(s) used to divide the figure. Separate subplot for level of the 'task' factor
    line_by = 'manner',                        # Within each suplot, do separate PRP lines for each manner category (averaged across phonemes of the same manner)
    subset_sensors = fronto_central_chs,       # Plot the average PRPs in this subset of electrodes
    equal_y_scale = True,                       # Ensure that the y-scale is the same across subplots
    show_topo = False         
    )

In [ ]:
from mne.viz import plot_topomap
from scipy.stats import zscore
from matplotlib.lines import Line2D

# =========================
# Settings
# =========================
fronto_central_chs = ['AF3', 'AF4', 'Fz', 'F3', 'F4', 'FC1', 'FC2']

time_windows = {
    "R1": (0.03, 0.06),
    "R2": (0.09, 0.15),
    "R3": (0.19, 0.26)
}

# time_windows = {
#     "R1": (0.14, 0.17),
#     "R2": (0.20, 0.26),
#     "R3": (0.30, 0.37)
# }

zscore_before_plot = True   # True = z-score PRP before plotting; False = raw PRP

# =========================
# Get data
# =========================
ds_all = prps.get_data()


# =========================
# Optional z-score
# =========================
if zscore_before_plot:
    x = ds_all['PRP'].x.copy()

    # PRP shape: case × time × sensor
    x_z = zscore(x, axis=1, nan_policy='omit')

    ds_all['PRP'] = eelbrain.NDVar(
        x_z,
        ds_all['PRP'].dims,
        name='PRP_z'
    )

    ylabel = 'PRP amplitude (z-score)'
    fig_suffix = 'zscore'
else:
    ylabel = 'PRP amplitude (µV)'
    fig_suffix = 'raw'

subs = sorted(set(str(s) for s in ds_all['sub']))

# =========================
# Compute subject-level average PRPs over FC sensors
# =========================
sub_waves = []
time_arr = None

for sub in subs:
    ds_sub = ds_all.sub(f"sub == '{sub}'")

    prp_fc = ds_sub['PRP'].sub(sensor=fronto_central_chs)

    nd = prp_fc.mean('case')
    y = nd.mean('sensor').x

    sub_waves.append(y)

    if time_arr is None:
        time = nd.time
        time_arr = np.linspace(time.tmin, time.tmax, time.nsamples)

sub_waves = np.asarray(sub_waves)
grand_mean = sub_waves.mean(axis=0)

# =========================
# Compute topographies for grand average PRP
# Use ALL sensors, not only FC sensors
# =========================
grand_nd = ds_all['PRP'].mean('case')

# =========================
# Figure layout
# =========================
fig = plt.figure(figsize=(10, 7))
gs = fig.add_gridspec(2, len(time_windows), height_ratios=[3, 1.2])

ax = fig.add_subplot(gs[0, :])

# =========================
# Shade selected time windows
# =========================
for label, (tmin, tmax) in time_windows.items():
    ax.axvspan(
        tmin,
        tmax,
        color='gray',
        alpha=0.18,
        linewidth=0
    )

    ax.text(
        (tmin + tmax) / 2,
        0.98,
        label,
        transform=ax.get_xaxis_transform(),
        ha='center',
        va='top',
        fontsize=10
    )

# =========================
# Plot individual subjects
# =========================
for y in sub_waves:
    ax.plot(
        time_arr,
        y,
        color='gray',
        alpha=0.45,
        linewidth=1.2
    )

# =========================
# Plot grand average
# =========================
ax.plot(
    time_arr,
    grand_mean,
    color='#D7255D',
    linewidth=3.5
)

ax.axvline(0, color='k', linestyle='--', linewidth=1)
# ax.axhline(0, color='k', linewidth=0.8)

ax.set_xlabel('Time (s)', fontsize=14)
ax.set_ylabel(ylabel, fontsize=14)
#ax.set_title('Average PRP: individual participants and grand average', fontsize=14)

legend_elements = [
    Line2D([0], [0], color='gray', lw=3.5, alpha=0.6, label='Individual participants'),
    Line2D([0], [0], color='#D7255D', lw=3.5, label='Grand average'),
]

# ax.legend(
#     handles=legend_elements,
#     loc='lower right',
#     frameon=False,
#     fontsize=12
# )

ax.tick_params(axis='both', labelsize=12)

# =========================
# Topographies for selected windows
# =========================
topo_axes = []

for i, (label, (tmin, tmax)) in enumerate(time_windows.items()):
    topo_ax = fig.add_subplot(gs[1, i])
    topo_axes.append(topo_ax)

    topo_nd = grand_nd.sub(time=(tmin, tmax)).mean('time')

    data = topo_nd.x
    sensor_pos = np.array([
        topo_nd.sensor.locs[s_idx][:2]
        for s_idx, _ in enumerate(topo_nd.sensor.names)
    ])

    im, _ = plot_topomap(
        data,
        sensor_pos,
        axes=topo_ax,
        show=False,
        contours=0
    )

    topo_ax.set_title(label, fontsize=11)

# =========================
# Shared colorbar
# =========================
cbar_ax = fig.add_axes([0.25, -0.1, 0.5, 0.02])

cbar = fig.colorbar(
    im,
    cax=cbar_ax,
    orientation='horizontal'
)
cbar.ax.tick_params(labelsize=10)

plt.tight_layout()

fig.savefig(
    f"{fig_out_dir}/NH_average_PRP_subjects_grand_topos_{fig_suffix}.png",
    dpi=900,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =========================
# Plot settings
# =========================
time_windows = {
    "90–120 ms": (0.09, 0.12),
    "140–240 ms": (0.14, 0.24),
    "600–850 ms": (0.60, 0.85),
}

title_fontsize = 18
axis_label_fontsize = 13
tick_fontsize = 11
legend_fontsize = 11
legend_title_fontsize = 12

legend_loc = "upper right"

# =========================
# Prepare data
# =========================
prp_data = prps.get_data()
prp_data['PRP_roi'] = prp_data['PRP'].sub(sensor=fronto_central_chs).mean('sensor')

manners = [
    ('Vowel', 'Vowel'),
    ('Stop', 'Stop'),
    ('Fricative', 'Fricative'),
    ('Nasal-approximant', 'Nasal-approximant')
]

conds = sorted(set(str(c) for c in prp_data['cond']))
subs = sorted(set(str(s) for s in prp_data['sub']))

# =========================
# Plot 2 x 2 panels
# =========================
fig, axes = plt.subplots(
    2, 2,
    figsize=(11, 7),
    sharex=True,
    sharey=True
)

axes = axes.ravel()

for ax, (manner_value, title) in zip(axes, manners):

    ds_manner = prp_data.sub(f"manner == '{manner_value}'")

    # -------------------------
    # Shade selected windows only
    # -------------------------
    for win_name, (tmin, tmax) in time_windows.items():
        ax.axvspan(
            tmin,
            tmax,
            alpha=0.12,
            color="gray",
            linewidth=0
        )

    # -------------------------
    # Plot condition waveforms
    # -------------------------
    for cond in conds:
        sub_waves = []
        time = None

        for sub in subs:
            ds_tmp = ds_manner.sub(
                f"(cond == '{cond}') & (sub == '{sub}')"
            )

            if len(ds_tmp) == 0:
                continue

            wave = ds_tmp['PRP_roi'].mean('case')
            sub_waves.append(wave.x)

            if time is None:
                time = wave.time.times

        if len(sub_waves) == 0:
            continue

        sub_waves = np.asarray(sub_waves)

        mean_wave = sub_waves.mean(axis=0)
        sem_wave = sub_waves.std(axis=0, ddof=1) / np.sqrt(sub_waves.shape[0])

        ax.plot(
            time,
            mean_wave,
            label=cond,
            linewidth=3
        )

        ax.fill_between(
            time,
            mean_wave - sem_wave,
            mean_wave + sem_wave,
            alpha=0.2
        )

    ax.axvline(0, color='k', linestyle='--', linewidth=0.8)
    # ax.axhline(0, color='k', linewidth=0.8)

    ax.set_title(title, fontsize=title_fontsize)
    ax.tick_params(axis='both', labelsize=tick_fontsize)

# =========================
# Labels and legend
# =========================
axes[0].set_ylabel('PRP amplitude (µV)', fontsize=axis_label_fontsize)
axes[2].set_ylabel('PRP amplitude (µV)', fontsize=axis_label_fontsize)
axes[2].set_xlabel('Time (s)', fontsize=axis_label_fontsize)
axes[3].set_xlabel('Time (s)', fontsize=axis_label_fontsize)

axes[1].legend(
    title='Condition',
    loc=legend_loc,
    fontsize=legend_fontsize,
    title_fontsize=legend_title_fontsize,
    frameon=False
)

plt.tight_layout()

fig.savefig(
    f"{fig_out_dir}/CI_prp_manner_emotion_plot.png",
    dpi=900,
    bbox_inches="tight"
)

plt.show()

## 6. F-statistic analysis across manner categories

This section computes and visualizes F-statistics for manner-related PRP differences. The line plots show how the F-statistic changes over time; the topographic plots show the spatial distribution at selected latencies.


### 6.1 Compute F-statistics

The F-statistic is computed separately by filename and tests differences across the `manner` categories.


In [ ]:
# Compute F
fstats = ns.FStatistic(
    prps,                   # Pass the PRPData object here
    by = 'filename',        # Default to 'filename', meaning that the results will be computed separately for each filename value
    category = 'manner'     # Defualt to 'manner'. Name of variable indicatings cateogry labels.
    )
fstats.save(f"{stat_out_dir}/NH_prp_manner_fstats.pkl")

### 6.2 Global F-statistic time course

A broad check of the manner effect over the fronto-central ROI.


In [ ]:
fstats.plot_f_stats(
    split_by = ['task'],
    subset_sensors = fronto_central_chs, 
    equal_y_scale = True,                        # Ensure that the y-scale is the same across subplots
    lw = 3.5,                                    # Linewidth
    show_topo = False
)

In [ ]:
# =========================
# Time windows to shade
# =========================
time_windows = {
    "R1": (0.03, 0.06),
    "R2": (0.09, 0.15),
    "R3": (0.19, 0.26),
    "R4": (0.35, 0.42)
}

# =========================
# Prepare data
# =========================
fstats_ds = fstats.get_data()

# subset sensors first, but do NOT average sensors yet
fstats_ds['F'] = fstats_ds['F'].sub(sensor=fronto_central_chs)

group_factors = ['task', 'stat']
agg_by = ' % '.join(group_factors)

drop_vars = [x for x in fstats_ds.keys() if x not in group_factors + ['F']]
fstats_avg = fstats_ds.aggregate(agg_by, drop=drop_vars)

# =========================
# Plot grand-average F-stat time course
# =========================
fig, ax = plt.subplots(figsize=(8, 5))

for stat in fstats_avg['stat'].cells:

    tmp = fstats_avg.sub(f"stat == '{stat}'")
    nd = tmp['F'][0]

    time = nd.time
    time_arr = np.linspace(time.tmin, time.tmax, time.nsamples)

    y = nd.get_data()

    # same as _plot_time_course: average sensors at plotting stage
    if y.shape[-1] > 1:
        y = np.mean(y, axis=-1)

    y = np.squeeze(y)

    ax.plot(
        time_arr,
        y,
        label=stat,
        linewidth=3.5,
        color='#F2545B'
    )

# =========================
# Shade and label selected time windows
# =========================
for label, (tmin, tmax) in time_windows.items():

    ax.axvspan(
        tmin,
        tmax,
        color='gray',
        alpha=0.18,
        linewidth=0
    )

    ax.text(
        (tmin + tmax) / 2,
        0.98,
        label,
        transform=ax.get_xaxis_transform(),
        ha='center',
        va='top',
        fontsize=11,
        color='black'
    )

# =========================
# Styling
# =========================
ax.axvline(0, color='k', linestyle='--', linewidth=1)
ax.axhline(0, color='k', linewidth=1)

ax.set_xlabel('Time (s)', fontsize=14)
ax.set_ylabel('F statistic', fontsize=14)
ax.set_ylim(0.4, 4)
ax.set_title('(FC) task=emo', fontsize=14)

ax.legend(frameon=False, fontsize=14)
ax.tick_params(axis='both', labelsize=12)

plt.tight_layout()

fig.savefig(
    f"{fig_out_dir}/NH_prp_manner_fstat_plot.png",
    dpi=900,
    bbox_inches="tight"
)

plt.show()

### 6.3 F-statistic by subject and condition

The marked latencies (`R1`–`R4`) are used consistently across F-statistic plots and topographies.


In [ ]:
# Mark these time points and their corresponding text labels
mark = {'time': [0.05, 0.12, 0.23, 0.4], 'label': ['R1', 'R2', 'R3', 'R4']}

# Plot the averaged PRP for each manner, divided by the factors in split_by
fstats.plot_f_stats(
    split_by = ['ses', 'cond'],
    subset_sensors = fronto_central_chs, 
    equal_y_scale = True,                        # Ensure that the y-scale is the same across subplots
    lw = 3.5,                                    # Linewidth
    mark = mark,                                 # Mark these time points
    show_topo = False
)

### 6.4 F-statistic by condition

This condition-level plot is useful for checking whether the manner effect differs between `hap` and `sad`.


In [ ]:
fstats.plot_f_stats(
    split_by = ['cond'],
    subset_sensors = fronto_central_chs, 
    equal_y_scale = True,                        # Ensure that the y-scale is the same across subplots
    lw = 3.5,                                    # Linewidth
    mark = mark,                                 # Mark these time points
    show_topo = False
)

### 6.5 F-statistic topographies by subject and condition

Topographic maps at selected latencies help identify where the manner-related effect is strongest.


In [ ]:
fstats.plot_f_topo(
    split_by= ['sub', 'cond'],
    times = [0.05, 0.12, 0.23, 0.4],
    cmap = 'Reds'
)

### 6.6 F-statistic topographies by condition

This provides the condition-level spatial summary at the same selected latencies.


In [ ]:
fstats.plot_f_topo(
    split_by= ['cond'],
    times = [0.05, 0.12, 0.23, 0.4],
    cmap = 'Reds'
)

## 7. Prepare dataset for Eelbrain statistical analysis

Retrieve the Eelbrain dataset from the `PRPData` object. Later cells add ROI-restricted variables and time-windowed versions of this dataset.


In [ ]:
prp_data = prps.get_data()

## 8. Exploratory sensor-level difference waves

This section plots subject-level `hap - sad` difference waves for vowel PRPs. Each butterfly line corresponds to one sensor, which helps identify whether condition differences are spatially consistent or driven by specific channels.


In [ ]:
ds_vowel = prp_data.sub("manner == 'Vowel'")
#ds_vowel = prp_data.sub("ses == '5'")
#ds_vowel['PRP'] = ds_vowel['PRP'].sub(sensor=['AF3', 'FC1', 'F3', 'FC5', 'F7', 'Fp1', 'T7', 'CP1'])

for sub in sorted(set(ds_vowel['sub'])):
    ds_sub = ds_vowel.sub(f"sub == '{sub}'")

    hap = ds_sub.sub("cond == 'hap'")['PRP'].mean('case')
    sad = ds_sub.sub("cond == 'sad'")['PRP'].mean('case')
    diff = hap - sad
    diff.name = f"{sub}: hap - sad"

    sensor_names = hap.get_dim('sensor').names

    for y, title in [
        (hap,  f"{sub}: vowel PRP hap, each sensor"),
        (sad,  f"{sub}: vowel PRP sad, each sensor"),
        (diff, f"{sub}: vowel PRP hap - sad, each sensor"),
    ]:
        p = eelbrain.plot.Butterfly(
            y,
            xlim=(-0.1, 1),
            axtitle=title
        )

        ax = p.axes[0]

        for line, sensor in zip(ax.lines, sensor_names):
            line.set_label(sensor)

        ax.legend(
            loc='upper right',
            bbox_to_anchor=(1.25, 1),
            fontsize=8
        )

## 9. Mass-univariate ANOVA over sensors and time

These ANOVAs test the effects of `cond`, `manner`, and their interaction while matching repeated observations by subject. The dependent variable is restricted to the fronto-central sensors but is not averaged over sensors unless explicitly done in later plotting cells.


### 9.1 Full-window ANOVA


In [ ]:
prp_data['PRP_roi'] = prp_data['PRP'].sub(sensor=fronto_central_chs).mean('sensor')
res = eelbrain.testnd.ANOVA(
    y = 'PRP_roi',
    x = 'cond * manner',
    match='sub',
    data = prp_data
)
print(res)
res.find_clusters(0.05)

### 9.2 Early-window ANOVA: 0–400 ms

This window targets the earlier PRP response period.


In [ ]:
prp_data_early = prp_data.copy()
prp_data_early['PRP'] = prp_data_early['PRP'].sub(time=(0, 0.4))
prp_data_early['PRP_roi'] = prp_data_early['PRP'].sub(sensor=fronto_central_chs)

res_early = eelbrain.testnd.ANOVA(
    y='PRP_roi',
    x='cond * manner',
    match='sub',
    data=prp_data_early
)
print(res_early)

### 9.3 Late-window ANOVA: 400–800 ms

This window targets later condition/manner effects after the early PRP response.


In [ ]:
prp_data_late = prp_data.copy()
prp_data_late['PRP'] = prp_data_late['PRP'].sub(time=(0.4, 0.8))
prp_data_late['PRP_roi'] = prp_data_late['PRP'].sub(sensor=fronto_central_chs)

res_late = eelbrain.testnd.ANOVA(
    y='PRP_roi',
    x='cond * manner',
    match='sub',
    data=prp_data_late
)
print(res_late)

## 10. ROI summaries for visualization

These plots average the fronto-central ROI when needed to summarize effects over time or within predefined time windows.


### 10.1 Condition effect over time in the ROI

The ROI is averaged over sensors, leaving a time course for visualizing condition effects.


In [ ]:
# Average ROI
prp_data['PRP_roi'] = prp_data['PRP'].sub(sensor=fronto_central_chs).mean('sensor')

# Emotion effect over time
eelbrain.plot.UTSStat(
    'PRP_roi',
    'cond',
    match='sub',
    data=prp_data
)

### 10.2 Early-window mean amplitude by manner

This collapses the early window over sensors and time, then visualizes the mean PRP amplitude by manner.


In [ ]:
# Early
prp_data_early['PRP_mean'] = (
    prp_data_early['PRP_roi']
    .mean('sensor')
    .mean('time')
)

eelbrain.plot.Barplot(
    y='PRP_mean',
    x='manner',
    match='sub',
    data=prp_data_early
)

### 10.3 Late-window mean amplitude by condition

This collapses the late window over sensors and time, then visualizes the mean PRP amplitude by condition.


In [ ]:
# Late
prp_data_late['PRP_mean'] = (
    prp_data_late['PRP_roi']
    .mean('sensor')
    .mean('time')
)

eelbrain.plot.Barplot(
    y='PRP_mean',
    x='cond',
    match='sub',
    data=prp_data_late
)

## 11. Planned pairwise follow-up tests

These tests compare `hap` vs. `sad` separately within specific manners.


### 11.1 Vowel PRP: `hap` vs. `sad`


In [ ]:
# Mass univariate test: emotion effect for vowel
ds_vowel = prp_data.sub("manner == 'Vowel'")
ds_vowel['PRP'] = ds_vowel['PRP'].sub(time=(0, 1))
#ds_vowel['PRP_R2'] = ds_vowel['PRP'].sub(time=(0.09, 0.12))
ds_vowel['PRP_roi'] = ds_vowel['PRP'].sub(sensor=fronto_central_chs)
#ds_vowel['PRP_roi_R2'] = ds_vowel['PRP_roi'].sub(time=(0.09, 0.12))
ds_vowel['PRP_roi_mean'] = ds_vowel['PRP_roi'].mean('sensor')
res_vowel = eelbrain.testnd.TTestRelated(
    'PRP_roi_mean',
    'cond',
    'hap',
    'sad',
    samples = 10000,
    match='sub',
    data=ds_vowel,
    pmin=0.05,
)
print(res_vowel)
p = eelbrain.plot.Butterfly(
    res_vowel
)

### 11.2 Stop PRP: `hap` vs. `sad`


In [ ]:
ds_stop = prp_data.sub("manner == 'Stop'")
ds_stop['PRP_roi'] = ds_stop['PRP'].sub(sensor=fronto_central_chs)
res_stop = eelbrain.testnd.TTestRelated(
    'PRP',
    'cond',
    'hap',
    'sad',
    match='sub',
    data=ds_stop,
    pmin=0.05,
)
print(res_stop)
p = eelbrain.plot.TopoButterfly(
    res_stop
)

### 11.3 Windowed Paired T test

In [ ]:
# Paired t-test for emotion effect in the early time window (0.09-0.12s)
# Define your time windows here
time_windows = {
    "N1 (90-120 ms)": (0.09, 0.12),
    "P2 (210-240 ms)": (0.21, 0.24),
    "LPC (600-850 ms)": (0.6, 0.85),
}

manners = [
    "Vowel",
    "Fricative",
    "Stop",
    "Nasal-approximant",
]

results = []

for manner in manners:
    for win_name, (tmin, tmax) in time_windows.items():

        ds_tmp = prp_data.sub(f"manner == '{manner}'").copy()

        ds_tmp['PRP_win'] = ds_tmp['PRP'].sub(time=(tmin, tmax))
        ds_tmp['PRP_roi'] = ds_tmp['PRP_win'].sub(sensor=fronto_central_chs).mean('sensor')
        ds_tmp['PRP_roi_mean'] = ds_tmp['PRP_roi'].mean('time')

        res = eelbrain.test.TTestRelated(
            'PRP_roi_mean',
            'cond',
            'hap',
            'sad',
            match='sub',
            data=ds_tmp,
        )

        results.append({
            "manner": manner,
            "window": win_name,
            "time_range_s": f"{tmin:.2f}-{tmax:.2f}",
            "t": res.t,
            "p": res.p,
        })

results_df = pd.DataFrame(results)
results_df

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import eelbrain

# =========================
# Plot settings
# =========================
time_windows = {
    "90-120 ms": (0.09, 0.12),
    "140-240 ms": (0.14, 0.24),
    "600-850 ms": (0.60, 0.85),
}

alpha_level = 0.05

title_fontsize = 18
axis_label_fontsize = 13
tick_fontsize = 11
legend_fontsize = 11
legend_title_fontsize = 12
star_fontsize = 18

legend_loc = "upper right"
# for outside legend, use:
# legend_loc = "upper left"
# legend_bbox = (1.02, 1)

# =========================
# Prepare data
# =========================
prp_data = prps.get_data()
prp_data['PRP_roi'] = prp_data['PRP'].sub(sensor=fronto_central_chs).mean('sensor')

manners = [
    ('Vowel', 'Vowel'),
    ('Stop', 'Stop'),
    ('Fricative', 'Fricative'),
    ('Nasal-approximant', 'Nasal-approximant')
]

conds = sorted(set(str(c) for c in prp_data['cond']))
subs = sorted(set(str(s) for s in prp_data['sub']))

# =========================
# Run paired t-tests
# =========================
test_results = {}

for manner_value, title in manners:
    test_results[manner_value] = {}

    for win_name, (tmin, tmax) in time_windows.items():

        ds_tmp = prp_data.sub(f"manner == '{manner_value}'").copy()

        ds_tmp['PRP_win'] = ds_tmp['PRP'].sub(time=(tmin, tmax))
        ds_tmp['PRP_roi_win'] = ds_tmp['PRP_win'].sub(sensor=fronto_central_chs).mean('sensor')
        ds_tmp['PRP_roi_mean'] = ds_tmp['PRP_roi_win'].mean('time')

        res = eelbrain.test.TTestRelated(
            'PRP_roi_mean',
            'cond',
            'hap',
            'sad',
            match='sub',
            data=ds_tmp,
        )

        test_results[manner_value][win_name] = {
            "t": res.t,
            "p": res.p,
            "sig": res.p < alpha_level,
            "time": (tmin, tmax),
        }

# =========================
# Plot 2 x 2 panels
# =========================
fig, axes = plt.subplots(
    2, 2,
    figsize=(11, 7),
    sharex=True,
    sharey=True
)

axes = axes.ravel()

for ax, (manner_value, title) in zip(axes, manners):

    ds_manner = prp_data.sub(f"manner == '{manner_value}'")

    # -------------------------
    # Shade t-test windows
    # -------------------------
    for win_name, result in test_results[manner_value].items():
        tmin, tmax = result["time"]

        if result["sig"]:
            ax.axvspan(tmin, tmax, alpha=0.30, color="orange")
            ax.text(
                (tmin + tmax) / 2,
                0.95,
                "*",
                transform=ax.get_xaxis_transform(),
                ha="center",
                va="top",
                fontsize=star_fontsize,
                color="orange"
            )
        else:
            ax.axvspan(tmin, tmax, alpha=0.12, color="gray")

    # -------------------------
    # Plot condition waveforms
    # -------------------------
    for cond in conds:
        sub_waves = []
        time = None

        for sub in subs:
            ds_tmp = ds_manner.sub(
                f"(cond == '{cond}') & (sub == '{sub}')"
            )

            if len(ds_tmp) == 0:
                continue

            wave = ds_tmp['PRP_roi'].mean('case')
            sub_waves.append(wave.x)

            if time is None:
                time = wave.time.times

        if len(sub_waves) == 0:
            continue

        sub_waves = np.asarray(sub_waves)

        mean_wave = sub_waves.mean(axis=0)
        sem_wave = sub_waves.std(axis=0, ddof=1) / np.sqrt(sub_waves.shape[0])

        ax.plot(
            time, 
            mean_wave, 
            label=cond,
            linewidth=3)
        ax.fill_between(
            time,
            mean_wave - sem_wave,
            mean_wave + sem_wave,
            alpha=0.2
        )

    ax.axvline(0, color='k', linestyle='--', linewidth=0.8)
    #ax.axhline(0, color='k', linewidth=0.8)

    ax.set_title(title, fontsize=title_fontsize)
    ax.tick_params(axis='both', labelsize=tick_fontsize)

# =========================
# Labels and legend
# =========================
axes[0].set_ylabel('PRP amplitude (µV)', fontsize=axis_label_fontsize)
axes[2].set_ylabel('PRP amplitude (µV)', fontsize=axis_label_fontsize)
axes[2].set_xlabel('Time (s)', fontsize=axis_label_fontsize)
axes[3].set_xlabel('Time (s)', fontsize=axis_label_fontsize)

axes[1].legend(
    title='Condition',
    loc=legend_loc,
    fontsize=legend_fontsize,
    title_fontsize=legend_title_fontsize,
    frameon=False
)

# Outside legend version:
# axes[1].legend(
#     title='Condition',
#     loc='upper left',
#     bbox_to_anchor=(1.02, 1),
#     fontsize=legend_fontsize,
#     title_fontsize=legend_title_fontsize,
#     frameon=False
# )

plt.tight_layout()
fig.savefig(
    f"{fig_out_dir}/NH_rp_manner_emotion_plot.png",
    dpi=900,
    bbox_inches="tight"
)
plt.show()